# AI Traffic Inspector: YOLOv8 Fine-Tuning & Evaluation

This notebook downloads an Indian Traffic dataset from Roboflow and fine-tunes a YOLOv8 model for detecting `helmet`, `no_helmet`, and `seatbelt` violations. 

It also automatically generates validation metrics (mAP, Precision, Recall, Confusion Matrix) that you can present to the Hackathon Judges to prove the mathematical accuracy of your custom model.

**Instructions:**
1. Go to **Runtime > Change runtime type** and ensure Hardware Accelerator is set to **T4 GPU**.
2. Run all cells.

In [ ]:
# 1. Install Dependencies
!pip install ultralytics roboflow -q
import ultralytics
ultralytics.checks()

In [ ]:
# 2. Download Indian Traffic Dataset from Roboflow
# Note: Replace YOUR_API_KEY with your free Roboflow API Key
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY_HERE")
project = rf.workspace("traffic-violations").project("indian-traffic-helmet-seatbelt")
dataset = project.version(1).download("yolov8")

print(f"Dataset downloaded to: {dataset.location}")

In [ ]:
# 3. Fine-Tune YOLOv8s on the Dataset
from ultralytics import YOLO

# Load a pretrained YOLOv8 small model
model = YOLO('yolov8s.pt')

# Train the model
# Note: We use 50 epochs for the hackathon demo, but 100+ is better for production.
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name='gridlock_yolov8_finetuned',
    device=0  # Use GPU
)

## System Evaluation Metrics (For Judges)
The cell below evaluates the fine-tuned model on the validation set and displays the Precision, Recall, and mAP metrics. It also plots the confusion matrix so you can show the judges exactly how well the model discriminates between `helmet` and `no_helmet`.

In [ ]:
# 4. Evaluate the Model (Generates Metrics for Judges)
val_results = model.val()

print("="*50)
print("🔥 AI TRAFFIC INSPECTOR MODEL PERFORMANCE METRICS 🔥")
print("="*50)
print(f"mAP50: {val_results.box.map50:.4f}")
print(f"mAP50-95: {val_results.box.map:.4f}")
print(f"Precision: {val_results.box.p.mean():.4f}")
print(f"Recall: {val_results.box.r.mean():.4f}")
print("="*50)

# Display Confusion Matrix
from IPython.display import Image, display
import glob

# Find the latest training run directory
latest_run = sorted(glob.glob('runs/detect/gridlock_yolov8_finetuned*'))[-1]

print("\nConfusion Matrix:")
display(Image(filename=f'{latest_run}/confusion_matrix.png', width=800))

print("\nF1-Confidence Curve:")
display(Image(filename=f'{latest_run}/F1_curve.png', width=800))

In [ ]:
# 5. Export Weights for Backend Integration
import shutil

# Copy the best weights to an accessible location
shutil.copy(f'{latest_run}/weights/best.pt', 'gridlock_finetuned_best.pt')
print("\n✅ Training Complete! Download 'gridlock_finetuned_best.pt' and place it in your backend/weights/ folder.")